# Puima Tutorial

In [3]:
from puima.collection_utils import DocumentCollection
from puima.pycas import PyCas
import puima.util as p_util

UIMA Idea: Collection Processing Engine consists of a Java class reading the documents, one or several components that process the CAS (add or remove annotations) based on some logic, and one component that serializes the CAS again.

In [6]:
coll = DocumentCollection("xmi_example")
for doc_name in coll.docs:
    doc = coll.docs[doc_name]
    # Do something with the document
    # Each document object is a PyCas object
    print(doc.get_language())
    # The document language was "x-unspecified", let's set it to English.
    doc.set_language("en")
    print("now:", doc.get_language())

# Serialize collection in case you made any changes.
coll.serialize("output_folder")

ignoring: TypeSystem.xml
ignoring: TypeSystemStyleMap.xml
x-unspecified
now: en


Puima does not retrieve the typesystem from TypeSystem.xml (yet).
You need to specify the class names of the annotation types manually in your code.

I chose `webanno.custom` as the prefix because this is what INCEpTION expects for custom types.
(This is because INCEpTION's predecessor system was called WebAnno.)

INCEpTION has performed the tokenization and sentence segmentation in this case.
HINT: If you want to use your own tokenization / segmentation, you can upload an XMI with your custom tokens to INCEpTION - simply upload the XMI instead of the raw text.

In [18]:
ANNOTATION = "uima.tcas.Annotation" # supertype for all annotations
SENTENCE = "de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence" # sentence type, created by INCEpTION using a simple rule-based sentence segmenter (afaik)
TOKEN = "de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Token" # sentence type, created by INCEpTION using a simple rule-based sentence segmenter (afaik)
TOY_ENTITY = "webanno.custom.ToyEntity" # our custom type

For demonstration purposes, we will now work with a single document.

In [8]:
for doc_name in coll.docs:
    doc = coll.docs[doc_name]
    break

# Now we have simply retrieved the first doc (just for tutorial purposes)

## Reading Annotations

Let us print out all the ToyEntity annotations.
In each iteration, toy_annot points to a different object of type `puima.pycas.Annotation`.
Observe how they are ordered by their begin character offsets.

<span style='font-size:32px;'>&#10071;</span> Puima does not re-implement the type structure in Python but maintains the types as strings that are attributes of the `Annotation` objects.
For writing bug-free client code, it is just important that you know exactly when you are working with an `Annotation` object and when you are working with a `string` (object) - more on this later.
Annotations are always `Annotation` instances, but feature values can either be `Annotation` or `string` objects.

In [14]:
for toy_annot in doc.select_annotations(TOY_ENTITY):
    print("\n", toy_annot) # For annotation objects, I always use the ending _annot -- makes my code easier to read!
    # What is printed is the output of the function __str__() of the Annotation class.
    # We can get the text of the string that is covered by the annotation like this:
    toy_annot_text = doc.get_covered_text(toy_annot)
    print("\ttext:", toy_annot_text)
    # We can obtain the string-based feature value of the annotation like this:
    label = toy_annot.get_feature_value("ToyType")
    print("\tlabel:", label)


 webanno.custom.ToyEntity begin:29 end:35
	text: rattle
	label: Baby & Toddler Toys

 webanno.custom.ToyEntity begin:130 end:140
	text: plush bear
	label: Dolls & Plush Toys

 webanno.custom.ToyEntity begin:216 end:225
	text: soft doll
	label: Dolls & Plush Toys

 webanno.custom.ToyEntity begin:309 end:315
	text: puzzle
	label: Games & Puzzles

 webanno.custom.ToyEntity begin:405 end:409
	text: ball
	label: Outdoor & Sports Toys

 webanno.custom.ToyEntity begin:510 end:519
	text: jump rope
	label: Outdoor & Sports Toys

 webanno.custom.ToyEntity begin:588 end:598
	text: plush bear
	label: Dolls & Plush Toys

 webanno.custom.ToyEntity begin:709 end:715
	text: puzzle
	label: Games & Puzzles

 webanno.custom.ToyEntity begin:787 end:791
	text: doll
	label: Dolls & Plush Toys

 webanno.custom.ToyEntity begin:906 end:912
	text: rattle
	label: Baby & Toddler Toys


<span style='font-size:32px;'>&#9997;</span> Exercise: Modify all the labels such that they do not contain any whitespace any more. Then serialize all the documents.

In [15]:
# Your code

The main advantage of using a framework like puima, cassis (or spacy) for working with annotated text is that you do not need to fiddle around with writing logic that checks annotation span overlaps etc. - the framework does that for you, can simple operate on the **logical level**, similar to how you would explain the idea to a colleague!

For example, we can ask puima to print out, for each ToyAnnotation, the sentence that "covers" it.
* At this point, you may wondering why we always call such functions on the document object.
* This is parallel to how the Java UIMA framework works.
* Recall that the CAS (document), i.e., our PyCas `doc`, contains all the data structures.
* Annotations are just helper objects (stand-off annotations) pointing to part of the document text.
* That is why these methods operate on the `doc` (need a reference to it). However, in this example, they also need to know the type of which they should retrieve annotations (`SENTENCE` in this case) and which annotation is supposed to be covered (hence the reference to `toy_annot`).
* Signature of the function in the `PyCas` class: `def select_covering(self, typename, annotation)`

In [19]:
for toy_annot in doc.select_annotations(TOY_ENTITY):
    print("\n", toy_annot)
    # Which sentence(s) cover the annotation?
    for sentence_annot in doc.select_covering(SENTENCE, toy_annot):
        print("\tsent:", doc.get_covered_text(sentence_annot))


 webanno.custom.ToyEntity begin:29 end:35
	sent: During the stormy night, the rattle on the shelf began shaking on its own, tapping out an irregular rhythm like a warning.

 webanno.custom.ToyEntity begin:130 end:140
	sent: Lina’s plush bear slowly lifted its stitched paw, pointing toward the dark hallway where her soft doll sat upright despite having been laid down earlier.

 webanno.custom.ToyEntity begin:216 end:225
	sent: Lina’s plush bear slowly lifted its stitched paw, pointing toward the dark hallway where her soft doll sat upright despite having been laid down earlier.

 webanno.custom.ToyEntity begin:309 end:315
	sent: On the floor, her half‑finished puzzle rearranged itself piece by piece, forming an image of eyes staring back at her.

 webanno.custom.ToyEntity begin:405 end:409
	sent: A lonely ball rolled out from under the bed, circling her feet as if urging her to follow.

 webanno.custom.ToyEntity begin:510 end:519
	sent: Outside, the abandoned jump rope slapped against 

If we expect exactly one covering annotation of the type:

In [20]:
for toy_annot in doc.select_annotations(TOY_ENTITY):
    print("\n", toy_annot)
    # Which sentence(s) cover the annotation?
    try:
        sentence_annot = next(doc.select_covering(SENTENCE, toy_annot))
        print("\tsent:", doc.get_covered_text(sentence_annot))
    except:
        print("Here you should handle the error properly!!")


 webanno.custom.ToyEntity begin:29 end:35
	sent: During the stormy night, the rattle on the shelf began shaking on its own, tapping out an irregular rhythm like a warning.

 webanno.custom.ToyEntity begin:130 end:140
	sent: Lina’s plush bear slowly lifted its stitched paw, pointing toward the dark hallway where her soft doll sat upright despite having been laid down earlier.

 webanno.custom.ToyEntity begin:216 end:225
	sent: Lina’s plush bear slowly lifted its stitched paw, pointing toward the dark hallway where her soft doll sat upright despite having been laid down earlier.

 webanno.custom.ToyEntity begin:309 end:315
	sent: On the floor, her half‑finished puzzle rearranged itself piece by piece, forming an image of eyes staring back at her.

 webanno.custom.ToyEntity begin:405 end:409
	sent: A lonely ball rolled out from under the bed, circling her feet as if urging her to follow.

 webanno.custom.ToyEntity begin:510 end:519
	sent: Outside, the abandoned jump rope slapped against 

<span style='font-size:32px;'>&#128128;</span> Actually, try avoiding the `select_covering` method as it will cause your code to become slow!

The much better recipe is to always go from larger annotation spans to smaller spans using `select_covered` as in the following example:

In [21]:
for sent_annot in doc.select_annotations(SENTENCE):
    print("\n", sent_annot)
    # Which toy annot(s) are within the sentene annotation span?
    for toy_annot in doc.select_covered(TOY_ENTITY, sent_annot):
        print("\toy:", doc.get_covered_text(toy_annot))
        print("\ttoy tokens:", [doc.get_covered_text(token_annot) for token_annot in doc.select_covered(TOKEN, toy_annot)])


 de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence begin:0 end:122
	oy: rattle
	toy tokens: ['rattle']

 de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence begin:123 end:276
	oy: plush bear
	toy tokens: ['plush', 'bear']
	oy: soft doll
	toy tokens: ['soft', 'doll']

 de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence begin:277 end:395
	oy: puzzle
	toy tokens: ['puzzle']

 de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence begin:396 end:486
	oy: ball
	toy tokens: ['ball']

 de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence begin:487 end:583
	oy: jump rope
	toy tokens: ['jump', 'rope']

 de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence begin:584 end:675
	oy: plush bear
	toy tokens: ['plush', 'bear']

 de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence begin:676 end:782
	oy: puzzle
	toy tokens: ['puzzle']

 de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence begin:783 end:863
	oy: doll
	toy tokens: ['doll'

## Adding Annotations (interactive)

Let us write a new annotator component that ADDs annotations.

* Let's create a new type `webanno.custom.Human` in INCEpTION.
* Download and export the data using the inception file util script.
* Implement Python code for marking all strings "Lina" and "her" as `Human`.
* Serialize the CAS again. Inspect with the AnnotationViewer.
* Upload in INCEpTION.